# Part 3: A/B Testing — Does the Apology Coupon Actually Work?
**⏱ This section takes approximately 60 minutes.**

---

## Scenario: Thursday — The Apology Coupon

By Thursday Sarah has two solid results to show:
- A distribution of polarity scores that reveals the true shape of customer sentiment
- A confidence interval around the model's 84% accuracy

Then Aisha Patel, Head of Customer Service, walks into the morning standup with an idea:

> *"Every customer the model tags as NEGATIVE — what if we automatically sent them an apology and a 10% coupon? I bet that turns some of them around."*

Sarah thinks this could work. But she has learned from the L01 workflow:
**framing** is the most important step. Before Aisha sends any coupons,
they need to test the idea properly.

That's when Marcus Wong, NorthStar's CTO, walks past and stops:

> *"Hold on. A 10% coupon on every negative review? That's thousands of pounds a week,*
> *plus engineering work to wire the trigger into our pipeline.*
> *Will this actually reduce churn — or are we adding ops complexity for noise?"*

This is his first real involvement in Sarah's project, and his question is the right one.
The apology coupon idea needs an experiment — not intuition.

**By the end of this notebook you will be able to:**
- Design a well-framed A/B test with clear hypotheses
- Run a simulated A/B test and compute a p-value from first principles
- Interpret a p-value correctly — including the three most common mis-readings
- Report results in a way that answers Marcus's question

In [ ]:
# Setup — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print("✅ Libraries loaded — you're ready to go!")


## Step 1 — Frame the Test

Before touching any data, Sarah writes down the four things that define a good A/B test.

| Component | Sarah's definition |
|---|---|
| **Null hypothesis (H₀)** | Sending the apology coupon has no effect on the first-30-days complaint rate |
| **Alternative hypothesis (H₁)** | Sending the apology coupon reduces the first-30-days complaint rate |
| **Outcome metric** | Complaint rate: proportion of customers who file a complaint within 30 days of purchase |
| **Two groups** | Control: NEGATIVE-tagged customers who receive no coupon; Treatment: NEGATIVE-tagged customers who receive the apology coupon |

Why complaint rate and not churn? Because complaint data is available in 30 days;
churn requires 90+ days of observation. Sarah and Marcus agree on 30-day complaint
rate as a reliable leading indicator.


In [ ]:
# Frame the test parameters
np.random.seed(42)

# Baseline complaint rate (control group — no coupon)
# About 18% of NEGATIVE-tagged customers file a formal complaint within 30 days
BASELINE_COMPLAINT_RATE = 0.18

# True effect of the coupon (what we set in simulation — the 'secret' we're trying to detect)
TRUE_EFFECT = -0.08   # The coupon reduces complaint rate by 8 percentage points

TRUE_TREATMENT_RATE = BASELINE_COMPLAINT_RATE + TRUE_EFFECT  # = 10%

# Sample size: NorthStar gets ~400 NEGATIVE-flagged customers per week
# Sarah and Marcus agree to run the test for 2 weeks per group
N_CONTROL   = 400
N_TREATMENT = 400

print("=== A/B Test Design ===")
print(f"Null hypothesis:         Coupon has zero effect on 30-day complaint rate")
print(f"Alternative hypothesis:  Coupon reduces complaint rate")
print(f"Baseline complaint rate: {BASELINE_COMPLAINT_RATE:.0%} (control group)")
print(f"Target to detect:        ≥5pp reduction (i.e., treatment rate ≤ {BASELINE_COMPLAINT_RATE-0.05:.0%})")
print(f"Sample per group:        {N_CONTROL} customers")
print(f"Total sample:            {N_CONTROL + N_TREATMENT} customers")
print()
print(f"(In simulation, true treatment rate = {TRUE_TREATMENT_RATE:.0%} — we set this to test detection)")

## Step 2 — Run the Experiment

Two weeks pass. Sarah's team assigns customers randomly:
- Half get the apology coupon
- Half get nothing

After 30 days, they count complaints.


In [ ]:
# Simulate the experiment results
np.random.seed(42)

# Control group: 400 customers, no coupon
control_complaints   = np.random.binomial(n=1, p=BASELINE_COMPLAINT_RATE, size=N_CONTROL)

# Treatment group: 400 customers, received coupon
treatment_complaints = np.random.binomial(n=1, p=TRUE_TREATMENT_RATE, size=N_TREATMENT)

# Summary
control_rate   = control_complaints.mean()
treatment_rate = treatment_complaints.mean()

print("=== Experiment Results (after 2 weeks) ===")
print()
print(f"  Control group (no coupon):")
print(f"    Customers: {N_CONTROL}")
print(f"    Complaints filed: {control_complaints.sum()}")
print(f"    Complaint rate: {control_rate:.2%}")
print()
print(f"  Treatment group (received coupon):")
print(f"    Customers: {N_TREATMENT}")
print(f"    Complaints filed: {treatment_complaints.sum()}")
print(f"    Complaint rate: {treatment_rate:.2%}")
print()
print(f"  Observed difference: {treatment_rate - control_rate:+.2%} ({treatment_rate:.2%} vs {control_rate:.2%})")
print()
print("Question: Is this difference REAL — or just random variation between the two groups?")


## ⏸️ Pause and Predict

We're about to run a statistical test to decide whether the observed difference
(between the control and treatment complaint rates) could have happened by chance.

**Before running the test, predict:**
- Marcus says: "A 5–6 percentage point difference sounds meaningful to me." Do you agree it's meaningful?
- Do you think this experiment, with 400 customers per group, will have enough data to detect the difference as statistically significant?
- What would it mean if the p-value came back as 0.08 (above 0.05)? Should Sarah recommend the coupon?

*Write your predictions here (double-click this cell to edit):*

## Step 3 — Test for Statistical Significance

We use a **two-proportion z-test** — the right tool for comparing complaint rates between two groups.

The formula matches the one in `lesson.md`: pool the two rates, compute the standard error
of the difference, then standardise. We will compute it from first principles using
`numpy` and the normal distribution from `scipy.stats` — the same formula you would
find in any statistics textbook.

In [ ]:
# Run the two-proportion z-test (computed from first principles)
control_complaints_n   = control_complaints.sum()
treatment_complaints_n = treatment_complaints.sum()

# Pooled proportion under the null hypothesis (no difference between groups)
p_pool = (control_complaints_n + treatment_complaints_n) / (N_CONTROL + N_TREATMENT)

# Standard error of the difference between the two rates
se = np.sqrt(p_pool * (1 - p_pool) * (1/N_CONTROL + 1/N_TREATMENT))

# Z-statistic: how many standard errors apart are the two observed rates?
z_stat = (treatment_rate - control_rate) / se

# One-sided p-value: probability of seeing a difference this large (or larger, in the
# expected direction) under the null hypothesis. We use the standard normal CDF.
p_value_z = st.norm.cdf(z_stat)

print("=== Statistical Test Results ===")
print()
print(f"  Control complaint rate:   {control_rate:.3%}")
print(f"  Treatment complaint rate: {treatment_rate:.3%}")
print(f"  Observed difference:      {treatment_rate - control_rate:+.3%}")
print()
print(f"  Pooled proportion:        {p_pool:.4f}")
print(f"  Standard error:           {se:.4f}")
print(f"  Z-statistic:              {z_stat:.3f}")
print(f"  P-value (one-sided):      {p_value_z:.4f}")
print()

alpha = 0.05
if p_value_z < alpha:
    print(f"  Result: STATISTICALLY SIGNIFICANT (p < {alpha})")
    print(f"  Conclusion: We reject the null hypothesis.")
    print(f"  The coupon appears to reduce complaint rates.")
else:
    print(f"  Result: NOT statistically significant (p >= {alpha})")
    print(f"  Conclusion: Insufficient evidence to reject the null hypothesis.")
    print(f"  We cannot conclude the coupon has a reliable effect at this sample size.")

### 💡 What do you notice?

- **The confidence intervals barely overlap** — this is a sign the effect is real and clearly visible. Completely overlapping CIs would suggest no meaningful difference.
- **Statistical significance vs practical significance** — the observed difference (~5–6 percentage points) is statistically significant (p < 0.05), but is it economically significant? That's Marcus's job to assess.
- **Sample size matters** — with 400 customers per group we are comfortably able to detect this size of effect. A smaller effect would have required a larger sample; a larger effect would have been detected with fewer customers.

**Back to our scenario:**
> Sarah can tell Marcus: "The coupon reduces 30-day complaint rates by approximately 5–6 percentage points. This result is statistically significant — it would occur by chance only [p-value × 100]% of the time if the coupon had no effect. Whether the cost of the coupon is justified by the reduction in complaints is a business decision for you."*

## Step 4 — The Three Most Common P-value Mis-readings

These mistakes appear constantly — in business, journalism, and even some published research.
Knowing them is part of being a responsible data analyst.

---

### ❌ Mis-reading 1: "p = 0.03 means there is a 3% chance the null hypothesis is true"

**What the p-value actually means:**
The p-value is the probability of seeing *results at least this extreme*, **assuming the null hypothesis is true**. It is not the probability that the null hypothesis is true.

Statistically: p = P(data | H₀), not P(H₀ | data).

The second form requires Bayesian inference — a different framework. See `optional_extensions.ipynb` for that.

**The practical correction:**
Instead of "3% chance the coupon doesn't work," say:
"If the coupon had zero effect, we would see a result this extreme only 3% of the time by chance."

---

### ❌ Mis-reading 2: "p < 0.05, so the effect is large and important"

**The truth:**
Statistical significance and practical (or economic) significance are separate.
With a large enough sample, *even a tiny meaningless difference* will produce p < 0.05.

**The practical correction:**
Always report the **effect size** alongside the p-value.
"The coupon reduced complaints by 2.5 percentage points (p = 0.031)" is far more informative than just "p = 0.031."

---

### ❌ Mis-reading 3: "p = 0.08, not significant — so the coupon doesn't work"

**The truth:**
p = 0.08 means you *failed to find enough evidence* to reject the null hypothesis — not that the null hypothesis is true. The effect might be real but undetectable at this sample size.

"Absence of evidence is not evidence of absence."

**The practical correction:**
"We did not find statistically significant evidence of an effect at the 0.05 level. We cannot conclude the coupon has no effect — the sample may have been too small to detect it reliably. To reduce uncertainty, we recommend extending the test or increasing the sample size."


## ✅ Section Summary

| Concept | What it means | Real-world use |
|---|---|---|
| **Null hypothesis** | The "boring" default: no effect, no difference | Always define it before running a test |
| **P-value** | Probability of seeing results this extreme if the null is true | p < 0.05: reject the null. p ≥ 0.05: insufficient evidence. |
| **Two-proportion z-test** | Compares rates between two groups | Complaint rates, conversion rates, churn rates |
| **Effect size** | How large the difference actually is | Report alongside p-value; significance ≠ importance |
| **Mis-reading 1** | p ≠ P(H₀ is true) | The p-value is about the data given H₀, not H₀ given the data |
| **Mis-reading 2** | Statistical ≠ practical significance | Large samples find tiny effects significant |
| **Mis-reading 3** | p > 0.05 ≠ "no effect" | Could be insufficient power, not absence of effect |

**Key insight for our scenario:**
> Sarah reports to Marcus: "The coupon reduced 30-day complaint rates by approximately 5–6 percentage points — a statistically significant result. The business decision of whether this justifies the coupon cost is yours, but the statistical evidence says the effect is real." Marcus nods. Priya nods. The model survives leadership review.

---

## 🏁 Friday: Three Numbers, Defended

Sarah walks into the Friday presentation with exactly three things:

| # | What she's reporting | Statistical tool |
|---|---|---|
| 1 | "60% of reviews are positive" | Distribution analysis — with the caveat about skew and threshold choice |
| 2 | "The model is 84% accurate (95% CI: 78%–89%)" | Confidence interval from 200 hand-labelled reviews |
| 3 | "The apology coupon reduced 30-day complaint rates by ~6pp (p < 0.05)" | A/B test with two-proportion z-test |

Priya nods. Marcus nods. The room is quiet.

Then Marcus says:

> *"OK. The sentiment model holds up. But you used a pre-trained one — off the shelf.*
> *Can you build us a model that predicts **churn** from our own customer data?*
> *Something trained on NorthStar behaviour, not on some generic internet corpus?"*

Sarah doesn't have an answer today.

That question — **can I train my own model?** — is the engine of **L03 (Supervised Learning)**.

---

**Next step →** Open `assignment.ipynb` to practise these skills and complete the after-class assignment.

*Or, if you want the deeper mathematical story: open `optional_extensions.ipynb` for Bayes' theorem, the t-test derivation, bootstrapping, and the CLT proof.*

---

## 🟡 Extension — self-study after class

*The cells below are optional. Skipping them will not affect your understanding of later lessons. Come back when you have time and want to go deeper — extra plots, bonus comparisons, and a few extra reflections that build on what you saw above.*

In [ ]:
# Visualise the results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: complaint rates
ax1 = axes[0]
groups = ['Control\n(no coupon)', 'Treatment\n(apology coupon)']
rates  = [control_rate, treatment_rate]
colors = ['steelblue', 'coral']
bars   = ax1.bar(groups, rates, color=colors, edgecolor='white', width=0.5, alpha=0.85)
ax1.axhline(BASELINE_COMPLAINT_RATE, color='grey', linestyle='--', linewidth=1.5,
            label=f'Baseline: {BASELINE_COMPLAINT_RATE:.0%}')
for bar, rate in zip(bars, rates):
    ax1.text(bar.get_x() + bar.get_width()/2, rate + 0.002, f'{rate:.1%}',
             ha='center', va='bottom', fontweight='bold')
ax1.set_ylabel('30-day Complaint Rate')
ax1.set_title(f'A/B Test Results\n(p = {p_value_z:.4f})')
ax1.set_ylim(0, 0.25)
ax1.legend()

# Confidence intervals for each rate
ax2 = axes[1]
for i, (rate, n, label, color) in enumerate([(control_rate, N_CONTROL, 'Control', 'steelblue'),
                                               (treatment_rate, N_TREATMENT, 'Treatment', 'coral')]):
    se = np.sqrt(rate * (1 - rate) / n)
    lower = rate - 1.96 * se
    upper = rate + 1.96 * se
    ax2.errorbar(x=i, y=rate, yerr=[[rate-lower], [upper-rate]],
                 fmt='o', color=color, capsize=8, capthick=2, markersize=10, linewidth=2,
                 label=f'{label}: {rate:.1%} (95% CI: {lower:.1%}–{upper:.1%})')

ax2.set_xlim(-0.5, 1.5)
ax2.set_xticks([0, 1])
ax2.set_xticklabels(['Control', 'Treatment'])
ax2.set_ylabel('30-day Complaint Rate with 95% CI')
ax2.set_title('Complaint Rates with Confidence Intervals')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
# Quantify the three mis-readings with concrete numbers from our test
print("=== Applying the Three Corrections to Sarah's Results ===")
print()
print(f"Test result: z = {z_stat:.3f}, p = {p_value_z:.4f}")
print()
print("❌ Mis-reading 1 (WRONG):")
print(f"   'There is a {p_value_z:.1%} chance the coupon doesn't work.'")
print()
print("✅ Corrected reading 1:")
print(f"   'If the coupon had zero effect on complaint rates, we would observe")
print(f"    a difference this large or larger only {p_value_z:.1%} of the time by chance alone.")
print(f"    We reject the hypothesis of no effect.'")
print()
print("❌ Mis-reading 2 (WRONG):")
print(f"   'p < 0.05 — the coupon has a significant (i.e. large) effect.'")
print()
print("✅ Corrected reading 2:")
print(f"   'The effect is statistically significant: {control_rate:.1%} vs {treatment_rate:.1%} complaint rate,")
print(f"    a {abs(treatment_rate - control_rate):.1%} reduction. Whether this {abs(treatment_rate - control_rate):.1%}")
print(f"    reduction justifies the coupon cost is a separate business decision.'")
print()
print("❌ Mis-reading 3 (WRONG, if p were 0.08):")
print("   'p > 0.05 — the coupon doesn't work. Case closed.'")
print()
print("✅ Corrected reading 3:")
print("   'We lack sufficient evidence to detect the effect at this sample size.")
print("    This does not mean the effect is zero. Recommend increasing sample size")
print("    before drawing conclusions.'")
